# Teaching Monster — Colab Pro+ GPU Notebook

**Runtime**: GPU (A100 on Pro+), High-RAM  
**Two modes** — pick the one you need:

| Mode | When | Cells to run |
|---|---|---|
| **A · One-shot video** | Test / local output | 0 → 10 |
| **B · API Server** | Competition / judges call your endpoint | 0 → 9, then 11 → 14 |

> Before running: **Runtime → Change runtime type → GPU → A100** (Pro+), enable High-RAM.

## 0 · GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU — change runtime to GPU first!"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU  : {gpu.name}")
print(f"VRAM : {gpu.total_memory / 1e9:.1f} GB")
print(f"torch: {torch.__version__}")

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Clone repository from GitHub

In [ ]:
import os
REPO_URL = "https://github.com/ragiokay/TeachingMonster-released-main.git"
REPO_DIR = "/content/repo"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!ls

## 3 · Install system packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libreoffice poppler-utils
print("System packages ready.")

## 4 · Install Python dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r requirements-colab-gpu.txt
print("Python packages ready.")

## 5 · Configure Gemini API key

In [ ]:
import getpass
from pathlib import Path

key = getpass.getpass("Paste your GEMINI_API_KEY: ")
Path("config/.env").write_text(f"GEMINI_API_KEY={key}\n", encoding="utf-8")
print("config/.env written.")

## 6 · Verify Gemini connectivity

In [ ]:
import yaml
from src.config_schema import AppConfig
from src.gemini_client import GeminiClient

cfg = AppConfig(**yaml.safe_load(open("config/default.yaml", encoding="utf-8")))
client = GeminiClient(cfg.llm)
client.load()
print("Gemini says:", client.generate("Reply with OK only.")[:80])

---
# Mode A · One-shot video generation
*(Skip to Mode B below if you're running the API server for the competition.)*

## 7A · Set topic

In [ ]:
REQUIREMENT_PROMPT = "Self-Attention Mechanism in Transformers"
PERSONA_PROMPT     = "High school student with no calculus background"
OUTPUT_NAME        = "teaching_monster_output.mp4"
print(f"Topic   : {REQUIREMENT_PROMPT}")
print(f"Audience: {PERSONA_PROMPT}")

## 8A · Run pipeline

In [ ]:
import os
os.environ.pop("TTS_FAST_MODE", None)
os.environ.pop("CURSOR_FAST_MODE", None)

!python -m scripts.T2V_pipeline \
  -r "{REQUIREMENT_PROMPT}" \
  -p "{PERSONA_PROMPT}" \
  -o "{OUTPUT_NAME}" \
  -d ./output

## 9A · Save to Google Drive

In [ ]:
DRIVE_OUT = "/content/drive/MyDrive/TeachingMonster/output"
!mkdir -p "{DRIVE_OUT}"
!cp -f ./output/{OUTPUT_NAME} "{DRIVE_OUT}/"
!ls -lh "{DRIVE_OUT}"
print(f"Saved to {DRIVE_OUT}/{OUTPUT_NAME}")

---
# Mode B · API Server (Competition mode)

The competition judges will POST to your server like this:

```json
POST /v1/video/generate
Content-Type: application/json

{
  "request_id": "job-001",
  "course_requirement": "Explain Newton's Second Law of Motion",
  "student_persona": "High school student, knows basic algebra"
}
```

Your server responds with:

```json
{
  "video_url": "https://xxxx.ngrok-free.app/v1/files/job-001/video",
  "subtitle_url": "https://xxxx.ngrok-free.app/v1/files/job-001/subtitle",
  "supplementary_url": null
}
```

They then download the video from `video_url`.

**Steps: run cells 11 → 14 (after completing 0–6 above).**

## 11 · Install ngrok (exposes Colab server to the internet)

In [ ]:
!pip -q install pyngrok

import getpass
from pyngrok import ngrok, conf

# Get your free ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok_token = getpass.getpass("Paste your ngrok authtoken: ")
conf.get_default().auth_token = ngrok_token
print("ngrok configured.")

## 12 · Start API server + open ngrok tunnel

In [ ]:
import subprocess, time, os
from pyngrok import ngrok

os.environ.pop("TTS_FAST_MODE", None)
os.environ.pop("CURSOR_FAST_MODE", None)

# Open tunnel first to get the public URL
tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url
print(f"Public URL: {PUBLIC_URL}")

# Write it so the server picks it up as BASE_URL
os.environ["BASE_URL"] = PUBLIC_URL

# Start FastAPI server in background
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "src.server:app",
     "--host", "0.0.0.0", "--port", "8000"],
    env={**os.environ, "BASE_URL": PUBLIC_URL},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for models to load (this takes ~10-20 min on first run)
print("Waiting for server startup and model download...")
print("Watch logs in cell 13 while waiting.")
print(f"\n*** Your API endpoint: {PUBLIC_URL}/v1/video/generate ***")

## 13 · Watch server logs (run until you see "Pipeline loaded successfully!")

In [ ]:
import io

# Stream server output until pipeline is loaded
for line in io.TextIOWrapper(server_proc.stdout, encoding="utf-8"):
    print(line, end="")
    if "Pipeline loaded successfully" in line:
        print("\n=== Server ready to accept requests ===")
        break

## 14 · Test your endpoint (dry-run, no generation)

In [ ]:
import requests as req

resp = req.post(
    f"{PUBLIC_URL}/v1/video/generate",
    json={
        "request_id": "dry-run-001",
        "course_requirement": "Test",
        "student_persona": "Test",
    },
    headers={"X-Dry-Run": "true"},
    timeout=10,
)
print("Status:", resp.status_code)
print("Response:", resp.json())
print(f"\nShare this endpoint with the judges:")
print(f"  POST {PUBLIC_URL}/v1/video/generate")

---
### Troubleshooting

| Symptom | Fix |
|---|---|
| `CUDA out of memory` | Enable High-RAM runtime and rerun from cell 0 |
| Server stuck loading models | Models are ~35 GB total download — wait 20+ min |
| ngrok 401 / tunnel error | Check authtoken at dashboard.ngrok.com |
| 504 timeout from judges | Pipeline took >30 min — reduce slide count or use staged mode |
| `ModuleNotFoundError` after restart | Rerun cells 3 and 4 |

### Health check
```
GET {PUBLIC_URL}/health
```
Returns `{"status": "healthy", "pipeline_loaded": true, ...}`